[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IfimoAI/moju/blob/main/examples/Notebooks/moju_slab_cooling_arxiv.ipynb)

# Moju slab cooling benchmark (Path A)

Reproducible companion to the Moju arXiv paper: train a `[2, 32, 32, 32, 1]` PINN on 1D transient slab cooling, run Moju audit on training and eval grids, and verify against bundled paper reference outputs.

- Paper: [Moju arXiv draft](https://arxiv.org/abs/0000.0000) *(tbd)*
- Code: [IfimoAI/moju](https://github.com/IfimoAI/moju)

**Colab:** clone the repo so `reference/32x32x32_opt/` is available, or open this notebook from `examples/Notebooks/` in the repository.

## Install

In [ ]:
# Colab / fresh environment
!pip install -q "moju==1.1.0" optax


## Setup (Colab)


In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("/content/moju").exists():
    !git clone --depth 1 https://github.com/IfimoAI/moju.git /content/moju
if Path("/content/moju/examples/Notebooks").exists():
    os.chdir("/content/moju/examples/Notebooks")


## Imports and JAX precision

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import optax

from moju.piratio import Operators, Models, Groups, Laws
from moju.monitor import ResidualEngine, audit, implied_group_specs_for_laws, visualize, build_loss, export_monitor_log, monitor_log_export_to_bundle

In [ ]:
# Enable Jax's 64-bit floating-point and integer precision
jax.config.update("jax_enable_x64", True)

## Problem constants (§4.1)

In [ ]:
L = 0.1 # Domain length
Lc = L # Characteristic length. (if slab is cooled on both sides Lc = L/2 and Lc = L if cooled from one side)
k_solid = 200.0 # reference thermal conductivity
rho_ref = 2700.0 # reference density
cp = 900.0 # specific heat capacity
h = 500.00 # convection heat transfer coefficient
T_inf = 300.0 # ambient temperature or the temperature of the "free-stream" fluid surrounding your material
T_i = 500.0 # initial temperature of material
t_min = 1.0
t_max = 60.0

# Initialize results list for further analysis
log_results = []

## MLP architecture `[2, 32, 32, 32, 1]`

In [ ]:
def init_mlp(key, widths):
    """
    Build jax NN layers from scratch
    """
    params = []
    for m, n in zip(widths[:-1], widths[1:]):
        key, sub = jax.random.split(key)
        W = jax.random.normal(sub, (n, m)) * jnp.sqrt(2.0 / m)
        b = jnp.zeros((n,))
        params.append({"W": W, "b": b})
    return params
def mlp(params, tx):
    """
    Assemble NN model from basic layers
    """
    h = tx
    for layer in params[:-1]:
        h = jnp.tanh(h @ layer["W"].T + layer["b"])
    out = params[-1]
    return h @ out["W"].T + out["b"]
# Second order optimizer
optimizer = optax.lbfgs()

## Input scaling, fields, and derivatives

In [ ]:
_delta_T = T_i - T_inf
_T_mid = (T_i + T_inf) / 2.0
_alpha_mid = Models.thermal_diffusivity(k_solid, rho_ref, cp)
_pde_residual_scale = _delta_T * max(1.0 / (t_max - t_min), float(_alpha_mid / (L**2)))
def _coords_norm(t, x, L=Lc):
    """Map physical (t, x) to normalized inputs ``(tau, xi)`` for the MLP."""
    t = jnp.asarray(t)
    x = jnp.asarray(x)
    dt = t_max - t_min
    if t.ndim == 0 and x.ndim == 1:
        tau = jnp.broadcast_to((t - t_min) / dt, x.shape[:-1] + (1,))
        xi = x / L
    elif t.ndim == 1 and x.ndim == 2:
        tau = ((t - t_min) / dt)[:, None]
        xi = x / L
    else:
        tau = jnp.broadcast_to((t - t_min) / dt, x.shape[:-1] + (1,))
        xi = x / L
    return jnp.concatenate([tau, xi], axis=-1)
def theta_field(params, t, x):
    """Dimensionless temperature :math:`\\theta=(T-T_\\infty)/(T_i-T_\\infty)` in (0, 1)."""
    tx = _coords_norm(t, x)
    xi = tx[...,1]
    raw = mlp(params, tx)[..., 0]
    
    # enforce dirichlet condition at x=L
    out = jax.nn.sigmoid(raw)
    return jnp.squeeze(out) if out.ndim > 0 and out.size == 1 else out
def scalar_field(params, t, x):
    theta = theta_field(params, t, x)
    T = T_inf + _delta_T * theta
    return jnp.squeeze(T) if T.ndim > 0 and T.size == 1 else T
def T_t_batch(params, t, x):
    """Time derivative of temperature"""
    return Operators.time_derivative(scalar_field, params, t, x)
def T_xx_batch(params, t, x):
    """Spatial second derivative of temperature"""
    def body(ti, xi):
        return Operators.laplacian(lambda p, x_in: scalar_field(p, ti, x_in), params, xi)

    return jax.vmap(body)(t, x)
def T_x_batch(params, t, x):
    """Spatial derivative dT/dx at (t, x). t (N,), x (N, 1)."""

    def body(ti, xi):
        grad = Operators.gradient(lambda p, x_in: scalar_field(p, ti, x_in), params, xi)
        return grad[0] if grad.shape == (1,) else grad

    return jax.vmap(body)(t, x)

## PINN loss (PDE + IC + Neumann + Robin BCs)

In [ ]:
def ic_uniform_slab(params, x_ic, t_ic, T_hot, lambda_ic=1.0):
    """
    Enforce T(x, t_ic) ≈ T_hot (uniform initial temperature).
    t_ic shape (n_ic, 1) broadcast; x_ic shape (n_ic, 1) or (nx, 1) grid.
    """
    Tt0 = scalar_field(params, t_ic, x_ic)           # whatever your forward is
    ref = jnp.asarray(float(T_hot))
    scale_T = jnp.maximum(jnp.abs(_delta_T), 1e-12)
    return lambda_ic * jnp.mean(((Tt0 - T_hot) / scale_T) ** 2)
def physics_loss_interior(params, t, x):
    T = scalar_field(params, t, x)
    kappa = k_solid
    rho_val = rho_ref
    alpha_loc = kappa / (rho_val * cp)
    return T_t_batch(params, t, x) - alpha_loc * T_xx_batch(params, t, x)
def neumann_loss_insulated_x0(params, t_bc, lambda_n=1.0, scale_n=None):
    scale_n = scale_n or (jnp.abs(_delta_T) / L)  # e.g. |ΔT|/L
    x0 = jnp.zeros((t_bc.shape[0], 1))
    Tx0 = T_x_batch(params, t_bc, x0)
    return lambda_n * jnp.mean((Tx0 / scale_n) ** 2)
def robin_bc_residual_xL(params, t_bc, h, k_L, T_amb):
    """
    Robin BC residual at x = L for 1D slab cooling:
        -k(T_L) * dT/dx|_{x=L} = h * (T_L - T_amb)
    Returns residual r(t):
        r = -k(T_L) * T_x(L) - h * (T_L - T_amb)
    Inputs
    ------
    params : model parameters
    t_bc   : jnp.ndarray, shape (N,)
    h      : scalar or array broadcastable to (N,)
    T_amb  : scalar or array broadcastable to (N,)
    """
    xL = jnp.full((t_bc.shape[0], 1), L)   # (N, 1)
    T_L = scalar_field(params, t_bc, xL)   # (N,)
    Tx_L = T_x_batch(params, t_bc, xL)     # (N,)
    # k_L = k_model(T_L)                     # (N,)
    return -k_L * Tx_L - h * (T_L - T_amb)
def robin_bc_loss_xL(params, t_bc, h, k_L, T_amb, lambda_r=1.0, scale_r=None):
    r = robin_bc_residual_xL(params, t_bc, h, k_L, T_amb)
    if scale_r is None:
        # nominal flux scale [W/m^2]
        scale_r = jnp.maximum(1e-12, jnp.abs(h) * jnp.maximum(1e-12, jnp.abs(_delta_T)))
    return lambda_r * jnp.mean((r / scale_r) ** 2)
def make_loss_fn(t_int, x_int, t_bc, x_ic, t_ic, h, k_L, T_hot, T_amb, lambda_ic=0.00001, lambda_n=0.0005, scale_n=None, lambda_r=0.00001, scale_r=None):
    def loss_fn(params):
        r_int = physics_loss_interior(params, t_int, x_int) / _pde_residual_scale
        loss_i = jnp.mean(r_int**2)
        loss_ic = ic_uniform_slab(params, x_ic, t_ic, T_hot, lambda_ic=lambda_ic)
        loss_n = neumann_loss_insulated_x0(params, t_bc, lambda_n=lambda_n)
        loss_r = robin_bc_loss_xL(params, t_bc, h, k_L, T_amb, lambda_r=lambda_r, scale_r=None)
        return loss_i + loss_ic + loss_n + loss_r
    return loss_fn
def make_train_step(loss_fn):
    """Return a jitted step that closes over ``loss_fn`` (uses module-level ``optimizer``)."""

    @jax.jit
    def train_step(params, opt_state):
        loss, grads = jax.value_and_grad(loss_fn)(params)
        updates, opt_state = optimizer.update(grads, opt_state, params,                           # current parameters
                                              value=loss,                       # scalar loss at current params
                                              grad=grads,                       # grad (same as updates before any transform)
                                              value_fn=loss_fn)
        params = optax.apply_updates(params, updates)
        return params, opt_state, loss

    return train_step

## Moju ResidualEngine and state builder

In [ ]:
engine_kw = {"constants": {"L": L, "cp": cp, "h": h, "k": k_solid, "rho": rho_ref, "alpha": Models.thermal_diffusivity(k_solid, rho_ref, cp)},
             "laws":[{"name": "fourier_conduction", "state_map": {"T_t": "T_t", "T_laplacian": "T_xx", "fo": "fo", "t": "t", "L": "L"}}],
             "groups": implied_group_specs_for_laws(["fourier_conduction"])}
def build_state_for_engine(params, t, x):
    T = scalar_field(params, t, x)
    T_t = T_t_batch(params, t, x)
    T_x = T_x_batch(params, t, x)
    T_xx = T_xx_batch(params, t, x)
    return {
        "T": T,
        "T_t": T_t,
        "T_x": T_x,
        "T_xx": T_xx,
        "t": t,
        "x":x
    }
# Initialize the residual engine for the training monitoring
train_engine = ResidualEngine(**engine_kw, best_effort_partial=False)

#Initialize a separate residual engine for
test_engine = ResidualEngine(**engine_kw, best_effort_partial=False)
def monitor_with_engine(params, t, x):
    """
    Builds state (generate predictions and their derivatives)
    from model params and the compute residuals
    """

    state_pred = build_state_for_engine(params, t, x)
    # Save state variables and derivatives for each training step
    log_results.append(state_pred)

    # Compute residuals
    residuals = train_engine.compute_residuals(state_pred, log_to_python=True)
    return build_loss(residuals)

## Collocation grids (training `64×48`, eval `512×384`)

In [ ]:
# set random state
key = jax.random.PRNGKey(0)
# Define domain discretization parameters
n_t, n_x = 64, 48

# Create flat collocation points in time and space
t_flat = jnp.linspace(t_min, t_max, n_t)
x_flat = jnp.linspace(0.0, L, n_x)

# --- Non-uniform transformation ---
power = 3.0  # Higher values add more points near t_min
# 1. Normalize t_flat to [0, 1]
normalized_t = (t_flat - t_min) / (t_max - t_min)
normalized_x = (x_flat - 0) / L
# 2. Apply power and scale back
t_flat = t_min + (t_max - t_min) * (normalized_t ** power)
x_flat = L * (normalized_x ** power)

# Reshape spatial and time coordinates
t_col, x_col = jnp.meshgrid(t_flat, x_flat, indexing="ij")
t_col = t_col.reshape(-1)
x_col = x_col.reshape(-1,1)

# Initial time and space collocation points
t_ic = jnp.full((n_x), float(t_min)).reshape(-1, 1)
x_ic = x_flat.reshape(-1, 1)

# Boundary time
t_bc = t_flat
dx, dt = L/n_x, t_max/n_t
# Compute losses and set up train step
loss_fn = make_loss_fn(t_col, x_col, t_bc=t_bc, x_ic=x_ic, t_ic=t_ic, T_hot=T_i, h=h, k_L=k_solid, T_amb=T_inf)
train_step = make_train_step(loss_fn)
train_engine.clear_log()
# Build the NN model and initialize the model weights
params = init_mlp(key, [2, 32, 32, 32, 1])
opt_state = optimizer.init(params)

## L-BFGS training (`14,000` steps, log every `200`)

In [ ]:
n_steps = 14000
# Run the training steps 
for step in range(n_steps):
    params, opt_state, loss = train_step(params, opt_state)
    if step % 200 == 0:
        law_loss = monitor_with_engine(params, t_col, x_col)
        print(f"step {step:4d}  loss={float(loss):.3e}  law_loss(engine)={float(law_loss):.3e}")

## Endpoint audit (eval grid)

In [ ]:
state_final = build_state_for_engine(params, t_col, x_col)
residuals_final = train_engine.compute_residuals(state_final, log_to_python=False)

# Eval grid (512 x 384 = 196,608 points)
# Collocation points for physics evaluation
n_t_phys, n_x_phys = n_t*8, n_x*8

t_flat_phys = jnp.linspace(1.0, t_max, n_t_phys)
x_flat_phys = jnp.linspace(0.0, L, n_x_phys)

t_col_phys, x_col_phys = jnp.meshgrid(t_flat_phys, x_flat_phys, indexing="ij")
t_col_phys = t_col_phys.reshape(-1)
x_col_phys = x_col_phys.reshape(-1,1)
state_pred = build_state_for_engine(params, t_col_phys, x_col_phys)
residuals_pred = test_engine.compute_residuals(state_pred, log_to_python=False)

train_report = audit(train_engine.log, last_residual_dict=residuals_final)
eval_report = audit(test_engine.log, last_residual_dict=residuals_pred)

print("Training overall:", train_report["overall_admissibility_score"], train_report["overall_admissibility_level"])
print("Training categories:", train_report["per_category"])
print("Eval overall:", eval_report["overall_admissibility_score"], eval_report["overall_admissibility_level"])
print("Eval categories:", eval_report["per_category"])
print("Eval per-key:", eval_report["per_key"])


## Visualize training trajectory and constitutive δ field

In [ ]:
train_export = export_monitor_log(
    train_engine.log,
    scope="visualize",
    mode="training",
    residuals=residuals_final,
    state_pred=state_final,
    persist=False,
)
eval_export = export_monitor_log(
    test_engine.log,
    scope="visualize",
    mode="eval",
    residuals=residuals_pred,
    state_pred=state_pred,
    persist=False,
)

fig_train = visualize(train_engine.log, residuals=residuals_final)
fig_train.show()

fig_eval = visualize(test_engine.log, mode="eval", backend="plotly", residuals=residuals_pred)
fig_eval.show()


## Verify against paper reference (`reference/32x32x32_opt/`)

In [ ]:
from pathlib import Path

REF = Path("reference/32x32x32_opt")
if not REF.exists():
    REF = Path("/content/moju/examples/Notebooks/reference/32x32x32_opt")

ref_adm = np.load(REF / "training_admissibility.npy", allow_pickle=True).item()
cat_path = REF / "category_scores.np.npy"
if not cat_path.exists():
    cat_path = REF / "category_scores.npy"
ref_cat = np.load(cat_path, allow_pickle=True).item()

train_bundle = monitor_log_export_to_bundle(train_export)
endpoint_cat = train_bundle["metrics"][-1]["category_admissibility_score"]
endpoint_keys = train_bundle["metrics"][-1]["admissibility_score"]

TOL_PP = 0.5  # percentage points on 0-100 scale

def pct(x):
    return 100.0 * float(x)

def assert_close(name, got, expected):
    delta_pp = abs(pct(got) - pct(expected))
    assert delta_pp <= TOL_PP, f"{name}: got {pct(got):.2f}%, expected {pct(expected):.2f}% (Δ={delta_pp:.2f} pp)"
    print(f"OK {name}: {pct(got):.2f}% (ref {pct(expected):.2f}%)")

assert_close("governing (laws)", endpoint_cat["laws"], ref_adm["governing admissibility"][-1])
assert_close("constitutive", endpoint_cat["constitutive"], ref_adm["constitutive admissibility"][-1])

for key, ref_val in ref_cat.items():
    got = endpoint_keys.get(key)
    assert got is not None, f"Missing key in endpoint audit: {key}"
    assert_close(key, got, ref_val)

print("Reference verification passed within", TOL_PP, "pp.")


## Maintainer export (optional)

In [ ]:
# Path B media bundles live under examples/Notebooks/media/data/.
# After any slab training run, export state dicts with a run/architecture prefix:
#
#   import sys
#   from pathlib import Path
#   sys.path.insert(0, str(Path("media").resolve()))  # or .../examples/Notebooks/media after git clone
#   from export_state_zips import DEFAULT_SLAB_PREFIX, export_state_bundle
#
#   # This notebook (32×32×32 opt)
#   export_state_bundle(
#       state_final,
#       state_pred,
#       "media/data",
#       prefix="32x32x32_opt_const_prop_1D_cooling_slab",
#   )
#
#   # Bundled w2 demo (default prefix)
#   export_state_bundle(state_final, state_pred, "media/data")  # prefix=DEFAULT_SLAB_PREFIX
